# Supplementary Figure 12 | Some novel lncRNAs mark previously annotated intergenic enhancers

**a.** Per-tissue percentage of genes whose transcription start site (TSS; ±500 bp) overlaps an ENCODE-rE2G active element, stratified by element class (promoter, genic enhancer, intergenic enhancer) and biotype. Points are per tissue (n = 12 with tissue-matched E2G predictions) and colored by tissue. Box plots show median and interquartile range (IQR); whiskers, 1.5× IQR. Some novel lncRNA TSSs overlap intergenic enhancers (median 6.5% across tested tissues).

**b.** Per-tissue percentage of nearby gene-protein-coding pairs (TSS within 1 Mb) whose QTLs colocalize in a joint ColocBoost model, stratified by focal gene biotype and whether the focal gene’s TSS is predicted as an E2G enhancer linked to that protein-coding gene. Points are per tissue and colored by GTEx tissue. Box plots show median and interquartile range (IQR); whiskers, 1.5× IQR.

**c.** Odds ratios for QTL colocalization given a TSS-linked-enhancer, from logistic regression of colocalization status on linked-enhancer status fit separately by focal gene biotype and by TSS-TSS genomic distance bins (<1 kb, 1-10 kb, 10-100 kb, 100 kb-1 Mb). Points, logistic-regression odds ratios (ORs); error bars, 95% CIs; asterisks, two-sided Wald p (* p < 0.05; ** p < 0.01; *** p < 0.001). Pairs with a TSS-linked-enhancer are more likely to colocalize, especially at longer distances.


## Setup


In [ ]:
from pathlib import Path

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from tqdm import tqdm

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.0)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

# Global NumPy random seed
np.random.seed(1)

control_window = 1_000_000
elem_groups = ["promoter", "genic", "intergenic"]
category_order = ["novel_lncRNA", "existing_lncRNA", "protein_coding"]
category_col = "gene_type_grouped"

enh_status_order = [True, False]
enh_status_labels = {
    True: "TSS is linked\nenhancer",
    False: "TSS is not linked\nenhancer",
}
dist_bin_edges = [0, 1_000, 10_000, 100_000, 1_000_001]
dist_bin_order = ["<1 kb", "1–10 kb", "10–100 kb", "100 kb–1 Mb"]

gene_type_palette = {
    "novel_lncRNA": sns.color_palette("Paired", 12)[5],
    "existing_lncRNA": sns.color_palette("Paired", 12)[4],
    "protein_coding": sns.color_palette("Paired", 12)[1],
}
cat_label = {
    "novel_lncRNA": "novel lncRNA",
    "existing_lncRNA": "existing lncRNA",
    "protein_coding": "protein coding",
}
promoter_flank = 500
e2g_cols = [
    "#chr", "start", "end", "class", "TargetGene", "TargetGeneEnsemblID",
    "isSelfPromoter", "Score", "normalizedDNase_enh.Feature",
]


def lnc_tss_targets_pc_enhancer(proms, e2g_by_target, pc):
    arr = e2g_by_target.get(pc)
    if arr is None or not proms:
        return False
    ec, es_arr, en_arr = arr
    for (pchrom, pstart, pend) in proms:
        if ((ec == pchrom) & (es_arr < pend) & (pstart < en_arr)).any():
            return True
    return False


## Paths

In [ ]:
gencode_class = Path("path/to/gencode_v39_v47_gene_classification.txt")
gtex_colors_json = Path("refs/colors.json")
e2g_by_tissue_parquet = Path("path/to/gene_tss_e2g_by_tissue.parquet")

e2g_dir = Path("path/to/e2g_predictions")
e2g_manifest = e2g_dir / "e2g_manifest.json"
lead_dir = Path("path/to/coloc_lncrna_leads")
annot_feather = Path("path/to/gencode.v47.annotation.gtf.feather")

fig_dir = Path("figures")
fig_pdf = fig_dir / "supp_figure_12_e2g.pdf"
fig_png = fig_dir / "supp_figure_12_e2g.png"


## Load data


### GTEx tissue colors


In [ ]:
with open(gtex_colors_json) as f:
    _colors = json.load(f)
    tcol = pd.DataFrame.from_dict(_colors, orient="index").reset_index(names="tissue_name")
tcol["tissue_color_hex"] = "#" + tcol["tissue_color_hex"]
tcol["tissue_file_name"] = (
    tcol["tissue_name"].str.replace("(", "", regex=False).str.replace(")", "", regex=False)
    .str.replace(" - ", "_").str.replace(" ", "_")
)
tissue_palette = dict(zip(tcol["tissue_file_name"], tcol["tissue_color_hex"]))
print(f"{len(tcol)} GTEx tissues in color table")


### Gene classification


In [ ]:
gclass = pd.read_table(gencode_class, low_memory=False)
gclass = gclass[gclass["gene_type_grouped"] != "artifact"]
new_lnc_ids = set(gclass.loc[gclass[category_col] == "novel_lncRNA", "gene_id_stable"])
existing_lnc_ids = set(gclass.loc[gclass[category_col] == "existing_lncRNA", "gene_id_stable"])
protein_coding_ids = set(gclass.loc[gclass[category_col] == "protein_coding", "gene_id_stable"])
lnc_ids_all = new_lnc_ids | existing_lnc_ids
lnc_category = {
    g: "novel_lncRNA" if g in new_lnc_ids else "existing_lncRNA" for g in lnc_ids_all
}
gene_category = {}
for ids, cat in [
    (new_lnc_ids, "novel_lncRNA"),
    (existing_lnc_ids, "existing_lncRNA"),
    (protein_coding_ids, "protein_coding"),
]:
    for gid in ids:
        gene_category[gid] = cat
print(
    f"{len(new_lnc_ids):,} new lncRNAs, {len(existing_lnc_ids):,} existing lncRNAs, "
    f"{len(protein_coding_ids):,} protein-coding genes"
)


### TSS–E2G overlap tables


In [ ]:
e2g_by_tissue = pd.read_parquet(e2g_by_tissue_parquet)
print(
    f"E2G TSS overlap table: {e2g_by_tissue['gene_stable'].nunique():,} genes × "
    f"{e2g_by_tissue['tissue'].nunique()} tissues"
)

### Gene annotation and TSS promoter windows


In [ ]:
annot = pd.read_feather(
    annot_feather,
    columns=["Chromosome", "Start", "End", "Strand", "Feature", "gene_id_nopatch", "gene_name", "gene_type"],
)
tx = annot[annot["Feature"] == "transcript"].copy()
tx["tss"] = np.where(tx["Strand"].values == "+", tx["Start"].values, tx["End"].values)
tx = tx.rename(columns={"Chromosome": "chrom", "gene_id_nopatch": "gene_stable"})
tss_table = tx[["gene_stable", "gene_name", "chrom", "Strand", "tss", "gene_type"]].reset_index(drop=True)
genes = annot[annot["Feature"] == "gene"].rename(
    columns={"Chromosome": "chrom", "gene_id_nopatch": "gene_stable"}
)[["gene_stable", "gene_name", "chrom", "Start", "End", "Strand", "gene_type"]].reset_index(drop=True)

prom_map = {}
for g, grp in tss_table[tss_table["gene_stable"].isin(lnc_ids_all)].groupby("gene_stable"):
    prom_map[g] = list(zip(
        grp["chrom"].to_numpy(),
        (grp["tss"] - promoter_flank).to_numpy(),
        (grp["tss"] + promoter_flank).to_numpy(),
    ))

pc_g = genes[genes["gene_stable"].isin(protein_coding_ids)].copy()
pc_g["tss"] = np.where(pc_g["Strand"].values == "+", pc_g["Start"].values, pc_g["End"].values)
pc_by_chrom = {}
for chrom, g in pc_g.groupby("chrom"):
    g = g.sort_values("tss")
    pc_by_chrom[chrom] = (g["tss"].to_numpy(), g["gene_stable"].to_numpy())
pc_tss = pc_g.drop_duplicates("gene_stable").set_index("gene_stable")[["chrom", "tss"]]

pc_prom_map = {}
for g, grp in tss_table[tss_table["gene_stable"].isin(protein_coding_ids)].groupby("gene_stable"):
    pc_prom_map[g] = list(zip(
        grp["chrom"].to_numpy(),
        (grp["tss"] - promoter_flank).to_numpy(),
        (grp["tss"] + promoter_flank).to_numpy(),
    ))

lnc_g = genes[genes["gene_stable"].isin(lnc_ids_all)].copy()
lnc_g["tss"] = np.where(lnc_g["Strand"].values == "+", lnc_g["Start"].values, lnc_g["End"].values)
lnc_tss = lnc_g.drop_duplicates("gene_stable").set_index("gene_stable")[["chrom", "tss"]]

print(f"{tss_table['gene_stable'].nunique():,} genes with transcript TSS")
print(f"{len(prom_map):,} lncRNAs with promoter windows (±{promoter_flank} bp)")
print(f"{len(pc_prom_map):,} protein-coding genes with promoter windows (±{promoter_flank} bp)")


### E2G prediction beds and coloc lead files


In [ ]:
with open(e2g_manifest) as fh:
    e2g_manifest = json.load(fh)

tissue_to_bed = {}
for m in e2g_manifest:
    tissue = m["gtex_tissue"]
    bed = e2g_dir / Path(m["file"]).name
    if not bed.is_file():
        raise FileNotFoundError(f"Missing E2G bed for {tissue}: {bed}")
    tissue_to_bed[tissue] = bed

leads_by_tissue = {}
for tissue in tissue_to_bed:
    lead_path = lead_dir / f"{tissue}.joint_gwas.all.leads.parquet"
    if not lead_path.is_file():
        raise FileNotFoundError(f"Missing coloc lead file for {tissue}: {lead_path}")
    leads_by_tissue[tissue] = pd.read_parquet(lead_path)

print(f"{len(tissue_to_bed)} tissues with E2G beds")
print(f"{len(leads_by_tissue)} tissues with coloc lead files")


### Compute — E2G links


In [ ]:
parts = []
for tissue, bed in tqdm(tissue_to_bed.items(), desc="E2G beds"):
    df = pd.read_table(bed, usecols=e2g_cols).rename(
        columns={
            "#chr": "chrom",
            "class": "elem_class",
            "normalizedDNase_enh.Feature": "dnase_enh_feature",
        }
    )
    df["tissue"] = tissue
    parts.append(
        df[["tissue", "chrom", "start", "end", "TargetGeneEnsemblID", "dnase_enh_feature"]]
    )
e2g_links = pd.concat(parts, ignore_index=True)


### E2G enhancer–gene links

In [ ]:
e2g_tissues = sorted(e2g_links["tissue"].dropna().unique())
print(f"{len(e2g_tissues)} tissues with E2G predictions")


### Compute — nearby lncRNA–PC pairs


In [ ]:
e2g_target_by_tissue = {}
for tissue, df in e2g_links.groupby("tissue", sort=True):
    e2g_target_by_tissue[tissue] = {
        t: (g["chrom"].to_numpy(), g["start"].to_numpy(), g["end"].to_numpy())
        for t, g in df.groupby("TargetGeneEnsemblID")
    }
bg_rows = []
for tissue in tqdm(e2g_tissues, desc="nearby controls"):
    leads = leads_by_tissue[tissue]
    sub = leads[leads["filtered_gene"].isin(lnc_ids_all)]
    if sub.empty:
        continue
    coloc_map = {}
    for r in sub.itertuples():
        partners = {str(g).split(".")[0] for g in (r.gene_ids if r.gene_ids is not None else [])}
        coloc_map.setdefault(r.filtered_gene, set()).update(partners & protein_coding_ids)
    e2g_by_target = e2g_target_by_tissue[tissue]
    for lnc in set(sub["filtered_gene"]):
        if lnc not in lnc_tss.index:
            continue
        cat = lnc_category[lnc]
        lchrom, ltss = lnc_tss.at[lnc, "chrom"], int(lnc_tss.at[lnc, "tss"])
        if lchrom not in pc_by_chrom:
            continue
        tss_arr, id_arr = pc_by_chrom[lchrom]
        lo = np.searchsorted(tss_arr, ltss - control_window)
        hi = np.searchsorted(tss_arr, ltss + control_window)
        coloc_pcs = coloc_map.get(lnc, set())
        proms = prom_map.get(lnc, [])
        for pc, ptss in zip(id_arr[lo:hi], tss_arr[lo:hi]):
            is_enh = lnc_tss_targets_pc_enhancer(proms, e2g_by_target, pc)
            bg_rows.append({
                "tissue": tissue, "lncRNA": lnc, "category": cat, "pc_gene": pc,
                "distance": abs(int(ptss) - ltss), "is_coloc": pc in coloc_pcs,
                "lncRNA_tss_enhancer": is_enh,
            })

bg_df = pd.DataFrame(bg_rows).drop_duplicates(["tissue", "lncRNA", "pc_gene"])
bg_df["group"] = np.where(bg_df["is_coloc"], "co-coloc", "nearby control")


### Nearby lncRNA–protein-coding gene pairs

In [ ]:
print(f"{len(bg_df):,} nearby (tissue, lncRNA, PC) pairs within {control_window:,} bp")


### Compute — nearby PC–PC pairs


In [ ]:
e2g_target_by_tissue = {}
for tissue, df in e2g_links.groupby("tissue", sort=True):
    e2g_target_by_tissue[tissue] = {
        t: (g["chrom"].to_numpy(), g["start"].to_numpy(), g["end"].to_numpy())
        for t, g in df.groupby("TargetGeneEnsemblID")
    }
pc_bg_rows = []
for tissue in tqdm(e2g_tissues, desc="PC–PC nearby controls"):
    leads = leads_by_tissue[tissue]
    coloc_map = {}
    pcs_in_tissue = set()
    for r in leads.itertuples():
        pcs = {str(g).split(".")[0] for g in (r.gene_ids if r.gene_ids is not None else [])}
        pcs &= protein_coding_ids
        if not pcs:
            continue
        pcs_in_tissue |= pcs
        for pc in pcs:
            coloc_map.setdefault(pc, set()).update(pcs - {pc})
    if not pcs_in_tissue:
        continue
    e2g_by_target = e2g_target_by_tissue[tissue]
    for pc in pcs_in_tissue:
        if pc not in pc_tss.index:
            continue
        lchrom, ltss = pc_tss.at[pc, "chrom"], int(pc_tss.at[pc, "tss"])
        if lchrom not in pc_by_chrom:
            continue
        tss_arr, id_arr = pc_by_chrom[lchrom]
        lo = np.searchsorted(tss_arr, ltss - control_window)
        hi = np.searchsorted(tss_arr, ltss + control_window)
        coloc_pcs = coloc_map.get(pc, set())
        proms = pc_prom_map.get(pc, [])
        for other, ptss in zip(id_arr[lo:hi], tss_arr[lo:hi]):
            if other == pc:
                continue
            is_enh = lnc_tss_targets_pc_enhancer(proms, e2g_by_target, other)
            pc_bg_rows.append({
                "tissue": tissue, "lncRNA": pc, "category": "protein_coding", "pc_gene": other,
                "distance": abs(int(ptss) - ltss), "is_coloc": other in coloc_pcs,
                "lncRNA_tss_enhancer": is_enh,
            })

pc_bg_df = pd.DataFrame(pc_bg_rows).drop_duplicates(["tissue", "lncRNA", "pc_gene"])
pc_bg_df["group"] = np.where(pc_bg_df["is_coloc"], "co-coloc", "nearby control")


### Nearby protein-coding–protein-coding gene pairs

In [ ]:
print(f"{len(pc_bg_df):,} nearby (tissue, PC, PC) pairs within {control_window:,} bp")


## Run analysis


### TSS overlap fractions by tissue


In [ ]:
per_tissue_rows = []
for tissue in tqdm(e2g_tissues, desc="TSS overlap %"):
    pg = e2g_by_tissue[e2g_by_tissue["tissue"] == tissue].copy()
    pg["category"] = pg["gene_stable"].map(gene_category)
    pg = pg[pg["category"].notna()]
    f = pg.groupby("category")[elem_groups].mean().reindex(category_order) * 100
    for cat in category_order:
        for grp in elem_groups:
            per_tissue_rows.append(
                {"tissue": tissue, "category": cat, "elem_group": grp, "pct": f.loc[cat, grp]}
            )
frac_per_tissue = pd.DataFrame(per_tissue_rows)
print(f"{len(frac_per_tissue):,} tissue × category × element rows")

### Co-coloc ~ enhancer within distance bins


In [ ]:
enrich_fit_inv = {}  # cat -> dist_bin -> OR stats
print("OR co-coloc ~ linked enhancer within distance bins:")
for cat, sub_df in [("novel_lncRNA", bg_df), ("existing_lncRNA", bg_df), ("protein_coding", pc_bg_df)]:
    sub = sub_df[sub_df["category"] == cat].copy()
    sub["enh_i"] = sub["lncRNA_tss_enhancer"].astype(int)
    sub["is_coloc_i"] = sub["is_coloc"].astype(int)
    sub["dist_bin"] = pd.cut(
        sub["distance"], bins=dist_bin_edges, labels=dist_bin_order,
        right=False, include_lowest=True,
    )
    enrich_fit_inv[cat] = {}
    for dbin in dist_bin_order:
        bin_sub = sub[sub["dist_bin"] == dbin]
        if bin_sub["enh_i"].nunique() < 2 or bin_sub["is_coloc_i"].nunique() < 2:
            raise ValueError(
                f"Cannot fit OR for {cat} / {dbin}: need both enhancer and coloc classes "
                f"(n={len(bin_sub):,}, enh={bin_sub['enh_i'].nunique()}, "
                f"coloc={bin_sub['is_coloc_i'].nunique()})"
            )
        fit = smf.logit("is_coloc_i ~ enh_i", data=bin_sub).fit(disp=False)
        orr = float(np.exp(fit.params["enh_i"]))
        ci = np.exp(fit.conf_int().loc["enh_i"])
        pval = float(fit.pvalues["enh_i"])
        enrich_fit_inv[cat][dbin] = {
            "OR": orr, "ci_low": float(ci.iloc[0]), "ci_high": float(ci.iloc[1]),
            "p": pval, "n": len(bin_sub),
        }
        print(
            f"  {cat:16s} {dbin:14s} OR = {orr:6.2f}  "
            f"(95% CI {ci.iloc[0]:.2f}-{ci.iloc[1]:.2f}), p = {pval:.2g}, n = {len(bin_sub):,}"
        )

source_c_inv = (
    pd.concat(
        [
            bg_df[["tissue", "category", "lncRNA_tss_enhancer", "is_coloc"]],
            pc_bg_df[["tissue", "category", "lncRNA_tss_enhancer", "is_coloc"]],
        ],
        ignore_index=True,
    )
    .groupby(["tissue", "category", "lncRNA_tss_enhancer"])["is_coloc"]
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"is_coloc": "pct_shared_qtl"})
)
print(f"source_c_inv: {len(source_c_inv):,} tissue × category × enhancer-status rows")


## Make figure


In [ ]:
fig = plt.figure(figsize=(7.2, 5.0), dpi=300)
gs = fig.add_gridspec(
    2, 6, height_ratios=[1.0, 1.15], hspace=0.55, wspace=0.55,
)

# a) TSS × E2G element overlap by category
for i, grp in enumerate(["promoter", "genic", "intergenic"]):
    ax = fig.add_subplot(gs[0, i * 2 : (i + 1) * 2])
    sub = frac_per_tissue[frac_per_tissue["elem_group"] == grp].copy()
    sns.boxplot(
        data=sub, x="category", y="pct", order=category_order, hue="category",
        fill=False, fliersize=0, linewidth=1.0, width=0.66,
        ax=ax, zorder=1, palette=gene_type_palette, legend=False,
    )
    sns.stripplot(
        data=sub, x="category", y="pct", order=category_order, hue="tissue",
        palette=tissue_palette, ax=ax, size=3.5, jitter=0.18, linewidth=0.35,
        edgecolor="k", zorder=2, legend=False,
    )
    titles = {"promoter": "promoter element", "genic": "genic enhancer", "intergenic": "intergenic enhancer"}
    pretty = {"novel_lncRNA": "novel lncRNA", "existing_lncRNA": "existing lncRNA", "protein_coding": "protein coding"}
    ax.set_title(titles[grp], fontsize=6)
    ax.set_xlabel("")
    ax.set_ylabel("% genes w/ TSS as active element" if i == 0 else "")
    ax.set_xticks(np.arange(len(category_order)))
    ax.set_xticklabels([pretty[c] for c in category_order], rotation=30, ha="right")
    ax.set_ylim(bottom=0)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    if i == 0:
        ax_a = ax

# b) % shared QTL | TSS linked-enhancer status + forest ORs
gs_b = gs[1, :].subgridspec(1, 2, width_ratios=[2.2, 1.2], wspace=0.35)
ax_b = fig.add_subplot(gs_b[0, 0])
ax_b_or = fig.add_subplot(gs_b[0, 1])

plot_cats = category_order
box_width = 0.7
sns.boxplot(
    data=source_c_inv, x="lncRNA_tss_enhancer", y="pct_shared_qtl", hue="category",
    order=enh_status_order, hue_order=plot_cats, palette=gene_type_palette,
    ax=ax_b, notch=False, fliersize=0, boxprops=dict(alpha=1), fill=False,
    width=box_width, gap=0.2,
)
n_hue = len(plot_cats)
each_w = box_width / n_hue
hue_xs = np.linspace(0, box_width - each_w, n_hue) - (box_width - each_w) / 2
hue_offsets = {cat: hue_xs[ci] for ci, cat in enumerate(plot_cats)}
rng = np.random.default_rng(1)
for gi, status in enumerate(enh_status_order):
    for cat in plot_cats:
        pts = source_c_inv[
            (source_c_inv["lncRNA_tss_enhancer"] == status) & (source_c_inv["category"] == cat)
        ]
        if pts.empty:
            continue
        x = gi + hue_offsets[cat] + rng.uniform(-0.04, 0.04, len(pts))
        ax_b.scatter(
            x, pts["pct_shared_qtl"].to_numpy(),
            c=[tissue_palette.get(t, "#999999") for t in pts["tissue"]],
            s=14, edgecolors="k", linewidths=0.35, zorder=3,
        )

y_max = float(source_c_inv["pct_shared_qtl"].max()) if len(source_c_inv) else 0.0
ax_b.set_xticks(range(len(enh_status_order)))
ax_b.set_xticklabels([enh_status_labels[s] for s in enh_status_order])
ax_b.set_xlabel("")
ax_b.set_ylabel("% pairs with shared QTL")
ax_b.set_ylim(0, max(y_max * 1.08, 1.0))
handles, labels = ax_b.get_legend_handles_labels()
ax_b.legend(
    handles[: len(plot_cats)], [cat_label.get(l, l) for l in labels[: len(plot_cats)]],
    title="", loc="upper right", frameon=False, ncol=1, fontsize=5,
)
for s in ("top", "right"):
    ax_b.spines[s].set_visible(False)

# c) forest: OR coloc ~ enhancer within distance bins (log2 axis, OR tick labels)
y_pos = np.arange(len(dist_bin_order))[::-1]
n_hue = len(plot_cats)
bin_offsets = np.linspace(-0.2, 0.2, n_hue)
for bi, dbin in enumerate(dist_bin_order):
    yi = y_pos[bi]
    for ci, cat in enumerate(plot_cats):
        res = enrich_fit_inv.get(cat, {}).get(dbin, {})
        orr, lo, hi = res.get("OR", np.nan), res.get("ci_low", np.nan), res.get("ci_high", np.nan)
        if pd.isna(orr) or orr <= 0 or lo <= 0 or hi <= 0:
            continue
        col = gene_type_palette[cat]
        x = np.log2(orr)
        y = yi + bin_offsets[ci]
        ax_b_or.errorbar(
            x, y,
            xerr=[[x - np.log2(lo)], [np.log2(hi) - x]],
            fmt="o", color=col, ecolor=col, elinewidth=1.0, capsize=0, markersize=4.5,
            markeredgecolor="k", markeredgewidth=0.5, zorder=3,
        )
        p = res.get("p", np.nan)
        if np.isfinite(p):
            if p < 0.001:
                stars = "***"
            elif p < 0.01:
                stars = "**"
            elif p < 0.05:
                stars = "*"
            else:
                stars = ""
            if stars:
                ax_b_or.text(
                    x, y + 0.05, stars,
                    ha="center", va="bottom", fontsize=4, color="black", zorder=4,
                )
ax_b_or.axvline(0.0, color="0.5", ls="--", lw=0.7, zorder=1)  # OR = 1
ax_b_or.set_yticks(y_pos)
ax_b_or.set_yticklabels(dist_bin_order)
# typical forest-plot style: log₂ positions, OR tick labels, axis labeled as odds ratio
ax_b_or.set_xlabel("Odds ratio\n(coloc ~ linked enhancer)")
ax_b_or.set_ylabel("")
_ci_high = [
    enrich_fit_inv.get(c, {}).get(b, {}).get("ci_high", np.nan)
    for c in plot_cats for b in dist_bin_order
]
_ci_low = [
    enrich_fit_inv.get(c, {}).get(b, {}).get("ci_low", np.nan)
    for c in plot_cats for b in dist_bin_order
]
x_hi_or = float(np.nanmax(_ci_high)) if np.any(np.isfinite(_ci_high)) else 2.0
x_lo_or = float(np.nanmin(_ci_low)) if np.any(np.isfinite(_ci_low)) else 1.0
x_hi_or = max(x_hi_or, 1.0)
pad = 0.2
# leave a little room left of OR=1 so the null line is not on the axis edge
x_lo_plot = min(np.log2(min(x_lo_or, 1.0)) - 0.12, -0.12)
x_hi_plot = np.log2(x_hi_or) + pad
ax_b_or.set_xlim(x_lo_plot, x_hi_plot)
k0 = int(np.floor(x_lo_plot))
k1 = int(np.ceil(x_hi_plot))
_or_ticks = [2 ** k for k in range(k0, k1 + 1) if k >= 0]  # 1, 2, 4, ...
ax_b_or.set_xticks([np.log2(v) for v in _or_ticks])
ax_b_or.set_xticklabels([f"{v:g}" for v in _or_ticks])
ax_b_or.set_ylim(-0.6, len(dist_bin_order) - 0.4)
_or_handles = [
    ax_b_or.plot(
        [], [], marker="o", color=gene_type_palette[c], markeredgecolor="k",
        markeredgewidth=0.5, linestyle="none", markersize=4.5, label=cat_label[c],
    )[0]
    for c in plot_cats
]
ax_b_or.legend(
    handles=_or_handles, loc="upper right",
    frameon=False, ncol=1, fontsize=5, handletextpad=0.3,
)
for s in ("top", "right", "left"):
    ax_b_or.spines[s].set_visible(False)
ax_b_or.tick_params(axis="y", length=0)

# Panel labels
fig.canvas.draw()
_inv = fig.transFigure.inverted()
_lab_x = min(
    _inv.transform(ax_a.transAxes.transform((-0.15, 0)))[0],
    _inv.transform(ax_b.transAxes.transform((-0.15, 0)))[0],
)
for ax, lab in [(ax_a, "a"), (ax_b, "b"), (ax_b_or, "c")]:
    _x = _lab_x if ax is not ax_b_or else _inv.transform(ax.transAxes.transform((-0.15, 0)))[0]
    _lab_y = _inv.transform(ax.transAxes.transform((0, 1.08)))[1]
    fig.text(_x, _lab_y, lab, fontweight="bold", va="top", ha="right", fontsize=9)

plt.show()


## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")


## Print stats


In [ ]:
# Median per-tissue % of novel lncRNA TSSs overlapping intergenic enhancers (panel a)
novel_intergenic = frac_per_tissue[
    (frac_per_tissue["category"] == "novel_lncRNA")
    & (frac_per_tissue["elem_group"] == "intergenic")
]
median_pct = float(novel_intergenic["pct"].median())
print(
    f"novel lncRNA TSS ∩ intergenic enhancer: median {median_pct:.1f}% "
    f"across {novel_intergenic['tissue'].nunique()} tissues"
)